In [1]:
!pip install pyspark

from pyspark import SparkContext
sc = SparkContext.getOrCreate()

In [2]:
logs = [
"Rahul login","Sneha login","Arjun login","Rahul purchase","Priya login",
"Sneha logout","Rahul login","Karan login","Arjun purchase","Sneha login",
"Rahul logout","Amit login","Priya purchase","Karan purchase","Sneha login",
"Rahul purchase","Meera login","Arjun login","Sneha purchase","Rahul login"
]
logs_rdd = sc.parallelize(logs)

sales = [
("Laptop",1,75000),("Mouse",3,500),("Keyboard",2,1500),("Laptop",1,75000),
("Monitor",1,12000),("Mouse",2,500),("Keyboard",1,1500),("Laptop",1,75000),
("Monitor",2,12000),("Mouse",3,500),("Laptop",1,75000),("Keyboard",2,1500),
("Monitor",1,12000),("Mouse",1,500),("Laptop",1,75000)
]
sales_rdd = sc.parallelize(sales)

employees = [
("Rahul","IT",70000),("Sneha","HR",60000),("Arjun","IT",75000),
("Priya","Finance",80000),("Karan","IT",50000),("Amit","HR",58000),
("Meera","Finance",82000),("Ravi","IT",72000),("Neha","HR",61000),
("Vikram","Finance",90000)
]
emp_rdd = sc.parallelize(employees)

In [3]:
logs_rdd.count()
logs_rdd.take(5)
logs_rdd.map(lambda x: x.split()[0]).collect()
logs_rdd.map(lambda x: x.split()[1]).distinct().collect()
logs_rdd.map(lambda x: x.split()[0]).distinct().collect()
logs_rdd.filter(lambda x: "login" in x).count()
logs_rdd.filter(lambda x: "purchase" in x).collect()

['Rahul purchase',
 'Arjun purchase',
 'Priya purchase',
 'Karan purchase',
 'Rahul purchase',
 'Sneha purchase']

In [4]:
sentences = sc.parallelize([
"big data spark",
"spark hadoop hive",
"python spark hadoop",
"spark streaming kafka"
])
words = sentences.flatMap(lambda x: x.split())
words.collect()
words.distinct().collect()
words.map(lambda x: (x,1)).reduceByKey(lambda a,b: a+b).collect()

[('big', 1),
 ('hadoop', 2),
 ('hive', 1),
 ('python', 1),
 ('streaming', 1),
 ('kafka', 1),
 ('data', 1),
 ('spark', 4)]

In [5]:
sales_rdd.map(lambda x: (x[0], x[1]*x[2])).collect()
sales_rdd.map(lambda x: x[1]*x[2]).reduce(lambda a,b: a+b)
sales_rdd.map(lambda x: x[0]).distinct().collect()
sales_rdd.filter(lambda x: x[1] > 1).collect()
sales_rdd.map(lambda x: (x[0], x[1])) \
         .reduceByKey(lambda a,b: a+b).collect()
sales_rdd.map(lambda x: (x[0], x[1]*x[2])) \
         .reduceByKey(lambda a,b: a+b).collect()
sales_rdd.map(lambda x: (x[0], x[1]*x[2])) \
         .reduceByKey(lambda a,b: a+b) \
         .sortBy(lambda x: x[1], ascending=False) \
         .take(1)

[('Laptop', 375000)]

In [6]:
emp_rdd.map(lambda x: x[0]).collect()
emp_rdd.filter(lambda x: x[2] > 70000).collect()
emp_rdd.map(lambda x: x[1]).distinct().collect()
emp_rdd.map(lambda x: (x[1], x[2])) \
       .reduceByKey(lambda a,b: a+b).collect()
emp_rdd.map(lambda x: (x[1], (x[2],1))) \
       .reduceByKey(lambda a,b: (a[0]+b[0], a[1]+b[1])) \
       .mapValues(lambda x: x[0]/x[1]).collect()
emp_rdd.sortBy(lambda x: x[2], ascending=False).take(1)

[('Vikram', 'Finance', 90000)]

In [7]:
emp_rdd.map(lambda x: (x[1],1)) \
       .reduceByKey(lambda a,b: a+b).collect()
emp_rdd.sortBy(lambda x: x[2], ascending=False).collect()
emp_rdd.map(lambda x: x[2]) \
       .sortBy(lambda x: -x).take(3)
emp_rdd.map(lambda x: (x[1], x[0])).collect()
emp_rdd.map(lambda x: (x[1], x[0])) \
       .groupByKey().mapValues(list).collect()

[('IT', ['Rahul', 'Arjun', 'Karan', 'Ravi']),
 ('HR', ['Sneha', 'Amit', 'Neha']),
 ('Finance', ['Priya', 'Meera', 'Vikram'])]

In [8]:
emp_rdd.map(lambda x: (x[1], x[2])) \
       .reduceByKey(lambda a,b: a+b) \
       .sortBy(lambda x: x[1], ascending=False) \
       .take(1)
emp_rdd.map(lambda x: (x[1], (1, x[2]))) \
       .reduceByKey(lambda a,b: (a[0]+b[0], a[1]+b[1])) \
       .collect()

[('IT', (4, 267000)), ('HR', (3, 179000)), ('Finance', (3, 252000))]

In [10]:
rev_per_product = sales_rdd.map(lambda x: (x[0], x[1]*x[2])) \
                           .reduceByKey(lambda a,b: a+b)
top_selling = sales_rdd.map(lambda x: (x[0], x[1])) \
                       .reduceByKey(lambda a,b: a+b) \
                       .sortBy(lambda x: x[1], ascending=False) \
                       .take(1)
highest_revenue = rev_per_product.sortBy(lambda x: x[1], ascending=False).take(1)
more_than_3 = sales_rdd.map(lambda x: (x[0],1)) \
                       .reduceByKey(lambda a,b: a+b) \
                       .filter(lambda x: x[1] > 3) \
                       .collect()
sorted_revenue = rev_per_product.sortBy(lambda x: x[1], ascending=False).collect()

rev_per_product.collect(), top_selling, highest_revenue, more_than_3, sorted_revenue

([('Laptop', 375000), ('Monitor', 48000), ('Mouse', 4500), ('Keyboard', 7500)],
 [('Mouse', 9)],
 [('Laptop', 375000)],
 [('Laptop', 5), ('Mouse', 4)],
 [('Laptop', 375000), ('Monitor', 48000), ('Keyboard', 7500), ('Mouse', 4500)])